# Clean EAL 2024 - Formación

Este notebook crea un dataset limpio de análisis a partir de los CSV procesados de la EAL 2024.

Criterio aplicado:
- No se conservan filas/cortes `TOTAL` como categoría de análisis.
- Se conservan los cortes acordados: `Más de 499 trabajadores`, `Transporte y almacenamiento`, `Servicios` cuando la tabla solo tiene sector agregado, y CCAA de sedes: `Cataluña`, `Madrid (Comunidad de)`, `Comunitat Valenciana`, `Andalucía`, `País Vasco`.
- Se transforma a formato largo para poder unir tablas con formatos diferentes.
- Al final se añaden columnas derivadas del título de tabla para conservar el significado que no queda explícito en otros campos.

In [5]:
import pandas as pd
from pathlib import Path
import re
import unicodedata


def encontrar_raiz_repo(inicio=None):
    ruta = Path(inicio or Path.cwd()).resolve()
    for candidata in [ruta, *ruta.parents]:
        if (candidata / "Equip_31").exists():
            return candidata
    raise FileNotFoundError("No se encontró la raíz del repositorio con Equip_31.")

REPO_ROOT = encontrar_raiz_repo()
PROCESSED_EAL = REPO_ROOT / "Equip_31/Data/Processed/EAL/2024"
OUTPUT_DIR = REPO_ROOT / "Equip_31/Data/clean/EAL"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "eal_2024_clean.csv"

ANIO = 2024

print("Repo:", REPO_ROOT)
print("Entrada:", PROCESSED_EAL)
print("Salida:", OUTPUT_CSV)

Repo: /Users/fedeur/ProjecteData
Entrada: /Users/fedeur/ProjecteData/Equip_31/Data/Processed/EAL/2024
Salida: /Users/fedeur/ProjecteData/Equip_31/Data/clean/EAL/eal_2024_clean.csv


## Funciones comunes

In [6]:
def limpiar_texto(valor):
    if pd.isna(valor):
        return ""
    valor = str(valor).replace("\n", " ")
    valor = re.sub(r"\s+", " ", valor)
    return valor.strip()


def quitar_acentos(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower().strip()


def leer_csv_hoja(hoja):
    ruta = PROCESSED_EAL / f"{hoja}.csv"
    if not ruta.exists():
        raise FileNotFoundError(f"No existe {ruta}")
    df = pd.read_csv(ruta, header=None, dtype=str, keep_default_na=False)
    df = df.map(limpiar_texto)
    df = df.replace("", pd.NA)
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    return df.fillna("").reset_index(drop=True)


def titulo_hoja(df, hoja):
    for valor in df.iloc[:, 0].tolist():
        texto = limpiar_texto(valor)
        if texto.upper().startswith(f"{hoja}."):
            return texto
    return ""


def denominador_hoja(df):
    for valor in df.iloc[:, 0].tolist():
        texto = limpiar_texto(valor)
        if texto.startswith("Año 2024."):
            return texto.replace("Año 2024. ", "")
    return ""


def hacer_columnas_unicas(columnas):
    resultado = []
    contador = {}
    for columna in columnas:
        columna = limpiar_texto(columna) or "SIN_TITULO"
        if columna not in contador:
            contador[columna] = 1
            resultado.append(columna)
        else:
            contador[columna] += 1
            resultado.append(f"{columna}_{contador[columna]}")
    return resultado


def extraer_codigo_eal16(texto):
    match = re.match(r"^(EAL-16[a-f]?)\.", limpiar_texto(texto), flags=re.IGNORECASE)
    return match.group(1).upper() if match else None


def fila_es_cabecera_eal16(row):
    valores = [limpiar_texto(x).upper() for x in row.tolist()]
    return {"TOTAL", "NADA", "POCO", "BASTANTE", "MUCHO"}.issubset(set(valores))


def info_titulo(hoja):
    mapa = {
        "EAL-16": ("Competencias", "Grado de importancia de competencias", "Total/tamaño/sector agregado"),
        "EAL-17": ("Contexto formativo", "Empresas según si impartieron formación", "Tamaño/actividad económica/CCAA"),
        "EAL-18": ("Necesidades formativas", "Detección de necesidades formativas y formación", "Tamaño/actividad económica/CCAA"),
        "EAL-18a": ("Medios", "Medios utilizados para proporcionar formación", "Tamaño/actividad económica/CCAA"),
        "EAL-18b": ("Suspensión o reducción", "Formación durante inactividad por suspensión o reducción de jornada", "Tamaño/actividad económica/CCAA"),
        "EAL-18c": ("Seguridad y salud", "Formación en seguridad, salud e higiene laboral", "Tamaño/actividad económica/CCAA"),
        "EAL-19": ("Contexto formativo", "Empresas formadoras por tamaño y sector agregado", "Cruce tamaño-sector agregado"),
        "EAL-20": ("Competencias", "Competencias en las que se formó", "Tamaño empresa"),
    }
    return mapa.get(hoja, ("", "", ""))


def agregar_info_titulo(df):
    df = df.copy()
    partes = df["Hoja"].map(lambda h: info_titulo(h))
    df["tema_titulo"] = partes.map(lambda x: x[0])
    df["indicador_titulo"] = partes.map(lambda x: x[1])
    df["desglose_publicado_titulo"] = partes.map(lambda x: x[2])
    return df

CATEGORIAS_ANALISIS = {
    "Más de 499 trabajadores",
    "Transporte y almacenamiento",
    "Servicios",
    "Cataluña",
    "Madrid (Comunidad de)",
    "Comunitat Valenciana",
    "Andalucía",
    "País Vasco",
}

SECCIONES = {
    "TAMAÑO DE LA EMPRESA": "Tamaño empresa",
    "ACTIVIDAD ECONÓMICA": "Sector",
    "COMUNIDAD AUTÓNOMA": "CCAA",
}

## Parsers de hojas

In [7]:
def parsear_eal16_clean():
    hoja = "EAL-16"
    df = leer_csv_hoja(hoja)
    titulo = titulo_hoja(df, hoja)
    denominador = denominador_hoja(df)

    registros = []
    titulo_actual = ""
    codigo_actual = None
    columnas_actuales = None
    tipo_desglose = ""
    grupo_desglose = ""

    for _, row in df.iterrows():
        primera = limpiar_texto(row.iloc[0])
        codigo = extraer_codigo_eal16(primera)
        if codigo:
            codigo_actual = codigo
            titulo_actual = primera
            columnas_actuales = None
            tipo_desglose = "Total"
            grupo_desglose = "Total empresas"
            if codigo == "EAL-16C":
                tipo_desglose = "Tamaño empresa"
                grupo_desglose = "Más de 499 trabajadores"
            elif codigo == "EAL-16F":
                tipo_desglose = "Sector agregado"
                grupo_desglose = "Servicios"
            elif codigo not in {"EAL-16", "EAL-16A", "EAL-16B", "EAL-16D", "EAL-16E"}:
                tipo_desglose = ""
                grupo_desglose = ""
            continue

        if codigo_actual and fila_es_cabecera_eal16(row):
            columnas_actuales = [limpiar_texto(x).upper() for x in row.tolist()]
            continue

        if not (codigo_actual and columnas_actuales):
            continue
        if grupo_desglose not in CATEGORIAS_ANALISIS:
            continue

        categoria = primera
        if categoria == "":
            continue

        for pos, variable in enumerate(columnas_actuales):
            if variable in {"", "TOTAL"}:
                continue
            valor = pd.to_numeric(limpiar_texto(row.iloc[pos]), errors="coerce")
            if pd.isna(valor):
                continue
            registros.append({
                "Año": ANIO,
                "Hoja": hoja,
                "Título de tabla": titulo_actual,
                "Categoría": categoria,
                "Variable": variable,
                "Valor": valor,
                "tipo_desglose": tipo_desglose,
                "grupo_desglose": grupo_desglose,
                "denominador_titulo": denominador,
            })
    return pd.DataFrame(registros)


def columnas_compatibles(hoja, df):
    if hoja == "EAL-17":
        fila_superior, fila_sub, start = 4, 5, 6
    elif hoja == "EAL-18":
        fila_superior, fila_sub, start = 4, 5, 6
    elif hoja == "EAL-18a":
        columnas = hacer_columnas_unicas(df.iloc[4, 1:].tolist())
        return dict(enumerate(columnas, start=1)), 5
    elif hoja == "EAL-18b":
        columnas = hacer_columnas_unicas(df.iloc[5, 1:].tolist())
        return dict(enumerate(columnas, start=1)), 6
    elif hoja == "EAL-18c":
        columnas = hacer_columnas_unicas(df.iloc[4, 1:].tolist())
        return dict(enumerate(columnas, start=1)), 5
    else:
        raise ValueError(hoja)

    superior = [limpiar_texto(x) for x in df.iloc[fila_superior].tolist()]
    sub = [limpiar_texto(x) for x in df.iloc[fila_sub].tolist()]
    columnas = {}
    grupo = ""
    for col in range(1, df.shape[1]):
        if superior[col]:
            grupo = superior[col]
        if col == 1 and hoja == "EAL-17":
            columnas[col] = "TOTAL"
        else:
            columnas[col] = f"{grupo} - {sub[col]}" if grupo and sub[col] else (grupo or sub[col])
    return columnas, start


def parsear_compatible_clean(hoja):
    df = leer_csv_hoja(hoja)
    titulo = titulo_hoja(df, hoja)
    denominador = denominador_hoja(df)
    columnas, start = columnas_compatibles(hoja, df)
    registros = []
    tipo_actual = "Total"

    for _, row in df.iloc[start:].iterrows():
        categoria = limpiar_texto(row.iloc[0])
        if categoria == "":
            continue
        cat_norm = categoria.upper()
        if cat_norm in SECCIONES:
            tipo_actual = SECCIONES[cat_norm]
            continue
        if categoria.startswith("("):
            continue
        if categoria not in CATEGORIAS_ANALISIS:
            continue

        for col, variable in columnas.items():
            if limpiar_texto(variable).upper() == "TOTAL":
                continue
            valor = pd.to_numeric(limpiar_texto(row.iloc[col]), errors="coerce") if col < len(row) else pd.NA
            if pd.isna(valor):
                continue
            registros.append({
                "Año": ANIO,
                "Hoja": hoja,
                "Título de tabla": titulo,
                "Categoría": categoria,
                "Variable": variable,
                "Valor": valor,
                "tipo_desglose": tipo_actual,
                "grupo_desglose": categoria,
                "denominador_titulo": denominador,
            })
    return pd.DataFrame(registros)


def parsear_eal19_clean():
    hoja = "EAL-19"
    df = leer_csv_hoja(hoja)
    titulo = titulo_hoja(df, hoja)
    denominador = denominador_hoja(df)
    columnas = hacer_columnas_unicas(df.iloc[4, 1:].tolist())
    registros = []
    for _, row in df.iloc[5:].iterrows():
        categoria = limpiar_texto(row.iloc[0])
        if categoria != "Más de 499 trabajadores":
            continue
        for offset, variable in enumerate(columnas, start=1):
            if variable != "SERVICIOS":
                continue
            valor = pd.to_numeric(limpiar_texto(row.iloc[offset]), errors="coerce")
            registros.append({
                "Año": ANIO,
                "Hoja": hoja,
                "Título de tabla": titulo,
                "Categoría": categoria,
                "Variable": variable,
                "Valor": valor,
                "tipo_desglose": "Cruce tamaño-sector agregado",
                "grupo_desglose": "Más de 499 trabajadores | Servicios",
                "denominador_titulo": denominador,
            })
    return pd.DataFrame(registros)


def parsear_eal20_clean():
    hoja = "EAL-20"
    df = leer_csv_hoja(hoja)
    titulo = titulo_hoja(df, hoja)
    denominador = denominador_hoja(df)
    columnas = hacer_columnas_unicas(df.iloc[4, 1:].tolist())
    registros = []
    for _, row in df.iloc[5:].iterrows():
        categoria = limpiar_texto(row.iloc[0])
        if categoria == "" or categoria.startswith("("):
            continue
        for offset, variable in enumerate(columnas, start=1):
            if variable != "MÁS DE 499 TRABAJADORES":
                continue
            valor = pd.to_numeric(limpiar_texto(row.iloc[offset]), errors="coerce")
            registros.append({
                "Año": ANIO,
                "Hoja": hoja,
                "Título de tabla": titulo,
                "Categoría": categoria,
                "Variable": "Más de 499 trabajadores",
                "Valor": valor,
                "tipo_desglose": "Tamaño empresa",
                "grupo_desglose": "Más de 499 trabajadores",
                "denominador_titulo": denominador,
            })
    return pd.DataFrame(registros)

## Crear clean

In [8]:
dfs_clean = [
    parsear_eal16_clean(),
    parsear_compatible_clean("EAL-17"),
    parsear_compatible_clean("EAL-18"),
    parsear_compatible_clean("EAL-18a"),
    parsear_compatible_clean("EAL-18b"),
    parsear_compatible_clean("EAL-18c"),
    parsear_eal19_clean(),
    parsear_eal20_clean(),
]

df_eal_2024_clean = pd.concat(dfs_clean, ignore_index=True)
df_eal_2024_clean = agregar_info_titulo(df_eal_2024_clean)

columnas_finales = [
    "Año",
    "Hoja",
    "Título de tabla",
    "Categoría",
    "Variable",
    "Valor",
    "tipo_desglose",
    "grupo_desglose",
    "denominador_titulo",
    "tema_titulo",
    "indicador_titulo",
    "desglose_publicado_titulo",
]

df_eal_2024_clean = df_eal_2024_clean[columnas_finales]

# Validaciones internas mínimas.
assert not df_eal_2024_clean.empty
assert not df_eal_2024_clean["Categoría"].str.upper().eq("TOTAL").any()
assert not df_eal_2024_clean["grupo_desglose"].str.upper().eq("TOTAL").any()
assert not df_eal_2024_clean["Variable"].str.upper().eq("TOTAL").any()
assert df_eal_2024_clean["Valor"].notna().all()

# Guardar salida clean.
df_eal_2024_clean.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("df_eal_2024_clean:", df_eal_2024_clean.shape)
print("CSV guardado en:", OUTPUT_CSV)

display(
    df_eal_2024_clean.groupby(["Hoja", "tipo_desglose", "grupo_desglose"], dropna=False)
    .size()
    .reset_index(name="n_registros")
)

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal_2024_clean)

df_eal_2024_clean: (260, 12)
CSV guardado en: /Users/fedeur/ProjecteData/Equip_31/Data/clean/EAL/eal_2024_clean.csv


,Hoja,tipo_desglose,grupo_desglose,n_registros
0,EAL-16,Sector agregado,Servicios,40
1,EAL-16,Tamaño empresa,Más de 499 trabajadores,40
2,EAL-17,CCAA,Andalucía,5
3,EAL-17,CCAA,Cataluña,5
4,EAL-17,CCAA,Comunitat Valenciana,5
5,EAL-17,CCAA,Madrid (Comunidad de),5
6,EAL-17,CCAA,País Vasco,5
7,EAL-17,Sector,Transporte y almacenamiento,5
8,EAL-17,Tamaño empresa,Más de 499 trabajadores,5
9,EAL-18,CCAA,Andalucía,6


,Año,Hoja,Título de tabla,Categoría,Variable,Valor,tipo_desglose,grupo_desglose,denominador_titulo,tema_titulo,indicador_titulo,desglose_publicado_titulo
0,2024,EAL-16,EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓX...,De dirección,NADA,1.219000,Tamaño empresa,Más de 499 trabajadores,Porcentaje sobre el total de empresas.,Competencias,Grado de importancia de competencias,Total/tamaño/sector agregado
1,2024,EAL-16,EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓX...,De dirección,POCO,8.240000,Tamaño empresa,Más de 499 trabajadores,Porcentaje sobre el total de empresas.,Competencias,Grado de importancia de competencias,Total/tamaño/sector agregado
2,2024,EAL-16,EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓX...,De dirección,BASTANTE,42.912000,Tamaño empresa,Más de 499 trabajadores,Porcentaje sobre el total de empresas.,Competencias,Grado de importancia de competencias,Total/tamaño/sector agregado
3,2024,EAL-16,EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓX...,De dirección,MUCHO,47.628000,Tamaño empresa,Más de 499 trabajadores,Porcentaje sobre el total de empresas.,Competencias,Grado de importancia de competencias,Total/tamaño/sector agregado
4,2024,EAL-16,EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓX...,De trabajo en equipo,NADA,0.350000,Tamaño empresa,Más de 499 trabajadores,Porcentaje sobre el total de empresas.,Competencias,Grado de importancia de competencias,Total/tamaño/sector agregado
5,2024,EAL-16,EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓX...,De trabajo en equipo,POCO,2.647000,Tamaño empresa,Más de 499 trabajadores,Porcentaje sobre el total de empresas.,Competencias,Grado de importancia de competencias,Total/tamaño/sector agregado
6,2024,EAL-16,EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓX...,De trabajo en equipo,BASTANTE,40.670000,Tamaño empresa,Más de 499 trabajadores,Porcentaje sobre el total de empresas.,Competencias,Grado de importancia de competencias,Total/tamaño/sector agregado
7,2024,EAL-16,EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓX...,De trabajo en equipo,MUCHO,56.333000,Tamaño empresa,Más de 499 trabajadores,Porcentaje sobre el total de empresas.,Competencias,Grado de importancia de competencias,Total/tamaño/sector agregado
8,2024,EAL-16,EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓX...,De atención al público/ trato a clientes,NADA,2.959000,Tamaño empresa,Más de 499 trabajadores,Porcentaje sobre el total de empresas.,Competencias,Grado de importancia de competencias,Total/tamaño/sector agregado
9,2024,EAL-16,EAL-16c. EMPRESAS DE MÁS DE 499 TRABAJADORES SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓX...,De atención al público/ trato a clientes,POCO,15.763000,Tamaño empresa,Más de 499 trabajadores,Porcentaje sobre el total de empresas.,Competencias,Grado de importancia de competencias,Total/tamaño/sector agregado
